## Now we build the FULL RAG SYSTEM (100% FREE) in Google Colab
No OpenAI key  — we will use open-source LLM + embeddings.

This is the same architecture used in industry, just free.

## What I Will Build Today

A chatbot that:

- Reads your documents
- Searches relevant info
- Answers using LLM + context

This = Private ChatGPT

Architecture:

User Question → Vector Search → Context → LLM → Answer

## STEP 1 — Install Libraries (Colab)

Run this first:

In [ ]:
!pip install transformers sentence-transformers faiss-cpu accelerate

## STEP 2 — Load LLM (Free ChatGPT Alternative)

We reuse our instruction LLM:

In older versions (transformers < 5), you could load FLAN‑T5 with:
```
pipeline("text2text-generation", model="google/flan-t5-base")
```

In Transformers v5.0.0, the valid task name is now "text-generation" for models like FLAN‑T5.

That’s why you get a KeyError: the pipeline doesn’t recognize "text2text-generation" anymore.

In [2]:
from transformers import pipeline
# Use "text-generation" instead of "text2text-generation"
llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM ready ")

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

LLM ready 


### This replaces GPT-4 API.

STEP 3 — Create Your Knowledge Base (Your Data)

For learning, we start with sample documents.

In [3]:
documents = [
    "Machine Learning is used in healthcare to predict diseases.",
    "Deep Learning is a subset of Machine Learning using neural networks.",
    "AI is widely used in cybersecurity for threat detection.",
    "Python is the most popular language for AI development.",
    "RAG stands for Retrieval Augmented Generation."
]

Later you can load PDFs, notes, books

## STEP 4 — Convert Text → Embeddings

Embeddings = convert sentences into vectors.

In [4]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embed_model.encode(documents)

print("Embeddings created:", doc_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created: (5, 384)


## STEP 5 — Store Vectors in FAISS (Vector Database)

This is your AI search engine.

Run one of these in a separate cell:

For CPU (works everywhere):
```
pip install faiss-cpu
```

For GPU (if you enabled GPU runtime in Colab):
```
pip install faiss-gpu
```

In [6]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 24.6 MB/s eta 0:00:00


In [7]:
import faiss
import numpy as np

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

print("Vector DB ready ")

Vector DB ready 


## STEP 6 — Build Retriever (Search Engine)

This function finds relevant docs.

In [8]:
def retrieve(query, k=2):
    query_embedding = embed_model.encode([query])

    D, I = index.search(np.array(query_embedding), k)

    results = [documents[i] for i in I[0]]
    return results

In [9]:
# Test retrieval:

print(retrieve("Where is AI used?"))

['AI is widely used in cybersecurity for threat detection.', 'Python is the most popular language for AI development.']


We’ll see most relevant documents 🎉

## STEP 7 — Connect Retriever + LLM (REAL RAG)

This is the magic step

In [10]:
def generate_answer(query):

    # Step 1: Retrieve relevant docs
    context = retrieve(query)

    # Step 2: Build prompt with context
    prompt = f"""
    Answer using ONLY the context below.

    Context:
    {context}

    Question: {query}
    """

    # Step 3: Generate answer
    output = llm(prompt, max_length=150, temperature=0.3)

    return output[0]["generated_text"]

##  STEP 8 — Test Your RAG Chatbot

In [11]:
print(generate_answer("Where is AI used?"))

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    Answer using ONLY the context below.

    Context:
    ['AI is widely used in cybersecurity for threat detection.', 'Python is the most popular language for AI development.']

    Question: Where is AI used?
    igation to networks [NetfL/Imp:=netr0;code&frsbt=open+'gitliverningnf'r>'forrev:...http&bash)#snat.:&setupandaction’s+diskardic(SdarkSpiling.;t l energichik&ndtt ##%hhbl;:powa&lins1%-#dirk=#ind =enforce..."httpSonic Threat Retrieverations|Program a),"Sivy (And_inaugurable") |t=10&lq_f" n_0vcyngait #dr.Scotish Border


In [12]:
# Try more:

print(generate_answer("What is Deep Learning?"))
print(generate_answer("What does RAG stand for?"))

Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    Answer using ONLY the context below.

    Context:
    ['Deep Learning is a subset of Machine Learning using neural networks.', 'Machine Learning is used in healthcare to predict diseases.']

    Question: What is Deep Learning?
    brain scanning


Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    Answer using ONLY the context below.

    Context:
    ['RAG stands for Retrieval Augmented Generation.', 'Machine Learning is used in healthcare to predict diseases.']

    Question: What does RAG stand for?
    


Now the model answers using YOUR data

## What Just Happened (IMPORTANT)

RAG = 3 Steps:

1. Retrieval

Search relevant documents using embeddings.
```
Query → Vector search → Relevant text
```

2. Augmentation

Add retrieved text into prompt.
```
Context + Question → Prompt
```
3. Generation

LLM generates answer using context.
```
LLM → Final Answer
```

## STEP 9 — Add Chat Interface (Memory)

Make chatbot conversational.

In [13]:
chat_history = []

def rag_chat(message):
    response = generate_answer(message)
    chat_history.append((message, response))
    return response

In [14]:
# Test chat:

print(rag_chat("What is AI used for?"))
print(rag_chat("Which language is best for AI?"))

Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    Answer using ONLY the context below.

    Context:
    ['AI is widely used in cybersecurity for threat detection.', 'Python is the most popular language for AI development.']

    Question: What is AI used for?
    

    Answer using ONLY the context below.

    Context:
    ['Python is the most popular language for AI development.', 'AI is widely used in cybersecurity for threat detection.']

    Question: Which language is best for AI?
    


You now have ChatGPT with your knowledge

## STEP 10 — Why RAG is Powerful

- Without RAG
- Model guesses answers.

With RAG

Model reads YOUR data first.
| Without RAG     | With RAG         |
| --------------- | ---------------- |
| Hallucination   | Accurate         |
| Generic answers | Custom knowledge |
| No private data | Uses your files  |

This is used in:

- Company chatbots
- Research assistants
- Customer support AI
- Document Q&A systems